# 207. Course Schedule
**Difficulty:** 🟡 Medium · **Topic:** Graph · **LeetCode:** https://leetcode.com/problems/course-schedule/

## 💡 Concepts

**Core concept(s):** Detect a **cycle** in a directed graph — via **DFS coloring** or **topological sort (Kahn's)**.

**Why it applies here:** Courses with prerequisites form a directed graph. You can finish all courses exactly when there's **no cycle** (no course indirectly requires itself). Both a DFS that spots a back-edge and a topological sort that can't place every node detect that cycle.

**Key intuition:** If you can order the courses so every prerequisite comes first, you're fine; a cycle makes that impossible.

---

### 📚 What is a Graph?
A **graph** is dots (**nodes/vertices**) joined by lines (**edges**). Edges can be **directed** (one-way, like prerequisites) or **undirected** (two-way, like friendships). A **grid** is just a graph where each cell links to its neighbors.
- **In Python:** usually an **adjacency list** — a `dict` mapping each node to the list of nodes it connects to.

### 📚 What is DFS (Depth-First Search)?
**DFS** follows one path as deep as it goes, then backtracks. On graphs you must remember **visited** nodes so you don't loop forever.
- **Complexity:** **O(V + E)** — each node and edge once.
- **In Python:** recursion or an explicit stack, plus a `visited` set.

### 📚 What is Topological Sort (Kahn's method)?
For a directed graph with no cycles, a **topological order** lists nodes so every arrow points forward (do prerequisites first). **Kahn's method:** repeatedly take a node with no remaining incoming arrows, output it, and remove its outgoing arrows.
- **Complexity:** **O(V + E)**. If you can't output every node, there's a **cycle**.
- **In Python:** an in-degree count per node + a queue of zero-in-degree nodes.

---

**Prerequisite knowledge:**
- Directed adjacency list.
- DFS visit-states, or in-degree counting.

## 📝 Problem

Given `numCourses` and `prerequisites` (each `[a, b]` = take `b` before `a`), return `True` if you can finish all courses.

**Example**
```
2, [[1,0]]        -> True
2, [[1,0],[0,1]]  -> False   (they need each other)
```

> Two approaches, both `O(V + E)`: DFS cycle detection and Kahn's topological sort.

### Approach 1 — DFS Cycle Detection

**Idea:** Color nodes: unvisited / in-progress / done. If DFS reaches an in-progress node, there's a cycle.

**Time:** `O(V + E)`. **Space:** `O(V + E)`.

In [ ]:
from collections import defaultdict, deque

def can_finish_dfs(numCourses, prerequisites):
    graph = defaultdict(list)              # course -> list of courses it depends on
    for a, b in prerequisites:
        graph[a].append(b)                 # a needs b done first
    state = [0] * numCourses               # 0 = unvisited, 1 = in-progress, 2 = safe/done
    def dfs(c):
        if state[c] == 1:
            return False                   # reached a course we're still exploring -> CYCLE
        if state[c] == 2:
            return True                    # already proven safe
        state[c] = 1                       # mark in-progress
        for nxt in graph[c]:               # every prerequisite must be completable
            if not dfs(nxt):
                return False
        state[c] = 2                       # this course (and its chain) is safe
        return True
    return all(dfs(c) for c in range(numCourses))

### Approach 2 — Kahn's Topological Sort

**Idea:** Repeatedly remove courses with no remaining prerequisites. If you place all of them, no cycle exists.

**Time:** `O(V + E)`. **Space:** `O(V + E)`.

In [ ]:
from collections import defaultdict, deque

def can_finish_kahn(numCourses, prerequisites):
    graph = defaultdict(list)              # course -> courses it unlocks
    indeg = [0] * numCourses               # how many prerequisites each course still has
    for a, b in prerequisites:
        graph[b].append(a)                 # finishing b unlocks a
        indeg[a] += 1
    q = deque([c for c in range(numCourses) if indeg[c] == 0])  # courses with no prereqs
    done = 0
    while q:
        c = q.popleft(); done += 1         # take a course with nothing blocking it
        for nxt in graph[c]:
            indeg[nxt] -= 1                # it no longer needs c
            if indeg[nxt] == 0:            # all its prereqs are met now
                q.append(nxt)
    return done == numCourses              # took every course -> no cycle

In [ ]:
# Correctness check
tests = [(2,[[1,0]],True), (2,[[1,0],[0,1]],False), (3,[[1,0],[2,1]],True), (3,[[0,1],[1,2],[2,0]],False)]
for n, pre, exp in tests:
    a, b = can_finish_dfs(n, pre), can_finish_kahn(n, pre)
    print(f"n={n}, pre={pre} -> dfs={a}, kahn={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

Inputs are shaped to force the worst case while keeping recursion shallow (stars / checkerboards) so nothing overflows the stack.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # every course depends on course 0 -> a valid (acyclic) star, shallow DFS
    pre = [[i, 0] for i in range(1, n)]
    return (n, pre)
solutions = {
    "dfs  O(V+E)": can_finish_dfs,
    "kahn O(V+E)": can_finish_kahn,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Cycle = impossible schedule:** ordering with dependencies is exactly topological sort; a cycle breaks it.
- **Two lenses:** DFS coloring finds a back-edge; Kahn's peels off zero-prerequisite nodes.
- **Signal:** "prerequisites / dependencies / ordering / can it be scheduled".
- **Related problems:** Course Schedule II (return the order), Alien Dictionary, Build System order.
- **Common pitfalls:** (1) reversing edge direction; (2) forgetting nodes with no edges still count.